# Experiment 7: Stacking vs LightGBM alone

Stacking: LightGBM + Logistic Regression as base learners, KNN as the meta-learner (5-fold CV).
The video keeps plain LightGBM because stacking scores about the same with a more complex model.
This notebook trains both on the same split so that decision rests on numbers rather than a claim.
LightGBM settings are the ones in the original `params.yaml` (class weights instead of SMOTE).
1,000 features, per experiment 3 (the original used 10,000 here).

In [ ]:
import os
from datetime import datetime

import mlflow
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI', 'http://127.0.0.1:5000'))
EXPERIMENT = 'Exp 7 - Stacking'
mlflow.set_experiment(EXPERIMENT)
BATCH = datetime.now().strftime('%Y%m%d-%H%M%S')

df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)

tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

In [ ]:
def make_lgbm():
    return LGBMClassifier(
        objective='multiclass',
        num_class=3,
        metric='multi_logloss',
        class_weight='balanced',
        reg_alpha=0.1,
        reg_lambda=0.1,
        learning_rate=0.08081298097796712,
        n_estimators=367,
        max_depth=20,
        random_state=42,
        verbose=-1,
    )


models = {
    'LightGBM': make_lgbm(),
    'Stacking_LGBM_LR_KNN': StackingClassifier(
        estimators=[
            ('lightgbm', make_lgbm()),
            ('logistic_regression', LogisticRegression(max_iter=1000, class_weight='balanced')),
        ],
        final_estimator=KNeighborsClassifier(n_neighbors=5),
        cv=5,
    ),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    with mlflow.start_run(run_name=name):
        mlflow.set_tags({'experiment_type': 'stacking', 'batch': BATCH})
        mlflow.log_metric('accuracy', accuracy_score(y_test, y_pred))
        for label, metrics in classification_report(y_test, y_pred, output_dict=True).items():
            if isinstance(metrics, dict):
                mlflow.log_metrics({f'{label}_{metric}': value for metric, value in metrics.items()})
    print(f'=== {name} ===')
    print(classification_report(y_test, y_pred))

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string=f"tags.batch = '{BATCH}'")
columns = {
    'tags.mlflow.runName': 'model',
    'metrics.accuracy': 'accuracy',
    'metrics.-1_recall': 'neg_recall',
    'metrics.macro avg_f1-score': 'macro_f1',
}
runs[list(columns)].rename(columns=columns).round(4)